## Calculate the sign concordance between RBP Activity Score and SHAP value within same cell line

In [1]:
# Load packages

import polars as pl
_=pl.Config.set_tbl_cols(100000)
_=pl.Config.set_tbl_rows(10000)
_=pl.Config.set_tbl_width_chars(10000)
_=pl.Config.set_fmt_str_lengths(10000)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, BoundaryNorm
import seaborn as sns

## Load necessary data

In [3]:
# Function to get path

from pathlib import Path

def get_path(cell_line: str, path_file: str = "data_path.txt") -> str:
    """
    Reads a base directory path from a text file and returns the full path
    to the cell line data directory.

    Args:
        cell_line (str): The name of the cell line (e.g. "K562" or "HepG2").
        path_file (str): Path to the text file containing the base directory path.

    Returns:
        str: Full path to the data file for the given cell line.
    """
    # Read base path from file
    base_path = Path(path_file).read_text().strip()
    
    # Build full path
    full_path = Path(base_path) / f"{cell_line}_all-data.feather"
    
    return str(full_path)

In [4]:
data_path_HepG2 = get_path("HepG2")
data_HepG2 = pl.read_ipc(data_path_HepG2)

Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.


In [5]:
data_path_K562 = get_path("K562")
data_K562 = pl.read_ipc(data_path_K562)

Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.


In [10]:
# Get RBP activity table

def plot_rbp_activity_heatmap(cell_line, data):
    
    print(cell_line)
    
    # Filter massive dataframe for KD samples, FDR <= 0.05, abs[DeltaPSI] >= 0.05
    filtered_data = (
        data
        .filter(
            (pl.col("Sample Name").str.contains("KD", literal=True))
            & (pl.col("FDR") <= 0.05)
            & (pl.col("DeltaPSI").abs() >= 0.05)
        )    
    )
    # Filter massive dataframe for KD samples, FDR <= 0.05, abs[DeltaPSI] >= 0.05

    filtered_data = (
        data
        .filter(
            (data["Sample Name"].str.contains("KD", literal=True))
            & (data["FDR"] <= 0.05)
            & (data["DeltaPSI"].abs() >= 0.05)
        )    
    )

    # Columns 
    columns_to_keep = [
        "RBP_KD_Target",
        "Sample Name",
        "FDR",
        "DeltaPSI",
        "has_RBP_KD_1",
        "has_RBP_KD_2",
        "has_RBP_KD_3",
        "has_RBP_KD_4",
        "has_RBP_KD_5",
        "has_RBP_KD_6",
        "Raw P-Val",
        "rMATS Event ID"
    ]
    
    rbp_targets = (
        filtered_data["RBP_KD_Target"]
        .unique()
        .sort()
        .to_list()
    )
    
    positions = [1,2,3,4,5,6]

    results = []  # collect all results here

    for rbp in rbp_targets:
        for pos in positions:

            rbp_specific_data = (
            filtered_data
            .filter(
                (pl.col("RBP_KD_Target") == rbp) & (pl.col(f"has_RBP_KD_{pos}") == True))
            .unique(subset=["rMATS Event ID"])) # gets in-silico KD rows per RBP per position

            # Drop unneccesary columns
            rbp_specific_data = rbp_specific_data.select(columns_to_keep)

            # Convert to pandas
            df = rbp_specific_data.to_pandas()

            # Prevents error for no data (SAFB or IGF..1)
            data = df["DeltaPSI"]

            if data.empty:
                continue 

            # DPSI conversion
            df["DeltaPSI"] = df["DeltaPSI"] * -1

            # Calculate the number of positive dPSI and the number of negative dPSI

            num_pos = (df["DeltaPSI"] > 0).sum()
            num_neg = (df["DeltaPSI"] < 0).sum()

            # Do not include rows plots with less than 6 sig. points 

            if (num_pos + num_neg) < 6:
                continue 

            # Normalized Difference Score
            norm_diff = (num_pos - num_neg) / (num_pos + num_neg)

            # Append results
            results.append({
                "RBP": rbp,
                "position": pos,
                "num_pos": num_pos,
                "num_neg": num_neg,
                "norm_diff": norm_diff
            })

    # Final results dataframe
    results_df = pd.DataFrame(results)
    
    # Can test for an individual RBP
    # test = results_df[results_df["RBP"] == ""]
    # print(test)
    
    df = pd.DataFrame(results_df)

    # Pivot the dataframe to have positions as columns, RBPs as rows, and score as values
    heatmap_data = df.pivot(index='RBP', columns='position', values='norm_diff')

    return heatmap_data

In [11]:
# Load RBP activity data
HepG2_data = plot_rbp_activity_heatmap('HepG2', data_HepG2)
K562_data = plot_rbp_activity_heatmap('K562', data_K562)

HepG2
K562


In [12]:
# Load glossary data
HepG2_glossary = pd.read_csv("/project/PlatigLab/users/reece/ENCODE-RNA-Binding-Protein-Network-Modeling/analysis/07_reece_biological_validations/1_shap_vs_dpsi_analysis/data/HepG2_global_SHAP_Signed-Local-SHAP-Mean-Bound-Only.tsv", sep="\t")
K562_glossary = pd.read_csv("/project/PlatigLab/users/reece/ENCODE-RNA-Binding-Protein-Network-Modeling/analysis/07_reece_biological_validations/1_shap_vs_dpsi_analysis/data/K562_global_SHAP_Signed-Local-SHAP-Mean-Bound-Only.tsv", sep="\t")

In [13]:
## Functions to calculate concordance with and without zeros

In [14]:
# Includes zeros and counts them as a match only if the other metric value is zero also

def calculate_sign_concordance(cell_line):
    
    # Manipulate dataframes
    print(cell_line)

    # Manipulate dataframes
    print(cell_line)
    t1 = globals()[f"{cell_line}_data"].copy()
    t2 = globals()[f"{cell_line}_glossary"].T.copy()
    t2.columns = t2.columns + 1  # shift 0-5 → 1-6 to match t1
    
    t1.index.name = None
    t1.columns.name = None
    t2.index.name = None
    t2.columns.name = None
    
    print("t1 columns:", t1.columns.tolist())  # [1, 2, 3, 4, 5, 6]
    print("t2 columns:", t2.columns.tolist())  # [1, 2, 3, 4, 5, 6]
    
    # heatmap_data: rows=RBP, cols=positions  (already correctly oriented)
    # table2: rows=positions, cols=RBP  →  transpose it
    
    # keep only RBPs present in both
    common_proteins = t1.index.intersection(t2.index)
    t1 = t1.loc[common_proteins]
    t2 = t2.loc[common_proteins]
    
    # align on positions
    t1, t2 = t1.align(t2, join="inner", axis=1)
    
    print(f"RBPs in common: {len(common_proteins)}")
    print(f"Positions being compared: {t1.shape[1]}")
    
    # Calculate % sign concordance: what % of the time are the signs the same?
    
    results = []
    total_match = 0
    total_valid = 0
    
    for protein in common_proteins:
        row1 = t1.loc[protein]
        row2 = t2.loc[protein]
    
        valid = row1.notna() & row2.notna()
        r1 = row1[valid]
        r2 = row2[valid]
    
        if len(r1) == 0:
            results.append({"protein": protein, "concordance": np.nan})
            continue
    
        matches = (np.sign(r1.values) == np.sign(r2.values)).sum()
        n        = len(r1)
        pct      = round(matches / n * 100, 2)
    
        total_match += matches
        total_valid += n
    
        results.append({"protein": protein, "concordance": pct})
    
    # outputs
    concordance_df = pd.DataFrame(results)
    total_concordance = round(concordance_df["concordance"].mean(), 2)
    
    print(f"\nTotal sign concordance across all proteins: {total_concordance}%\n")
    print(concordance_df.to_string(index=False))

In [9]:
K562 = calculate_sign_concordance('K562')

K562
K562
t1 columns: [1, 2, 3, 4, 5, 6]
t2 columns: [1, 2, 3, 4, 5, 6]
RBPs in common: 31
Positions being compared: 6

Total sign concordance across all proteins: 42.31%

protein  concordance
  AGGF1        66.67
 AKAP8L        66.67
    AQR        33.33
  BUD13        66.67
  DDX24         0.00
  DDX3X         0.00
 DROSHA         0.00
 EFTUD2        66.67
   FMR1        33.33
   FXR1        75.00
  GPKOW        66.67
 HNRNPC       100.00
 HNRNPK        50.00
 HNRNPM       100.00
IGF2BP2         0.00
  KHSRP        83.33
  PCBP1         0.00
  PTBP1        50.00
 RBFOX2         0.00
  RBM15        50.00
  RBM22       100.00
  SF3B4        33.33
 SMNDC1         0.00
  SRSF1        33.33
 TARDBP         0.00
   TIA1       100.00
  U2AF1        20.00
  U2AF2        33.33
  UCHL5        83.33
   UPF1         0.00
   YBX3         0.00


In [10]:
HepG2 = calculate_sign_concordance('HepG2')

HepG2
HepG2
t1 columns: [1, 2, 3, 4, 5, 6]
t2 columns: [1, 2, 3, 4, 5, 6]
RBPs in common: 32
Positions being compared: 6

Total sign concordance across all proteins: 51.67%

protein  concordance
  AKAP1       100.00
    AQR        33.33
 BCLAF1        50.00
  DDX3X        33.33
  DDX59         0.00
 EFTUD2        40.00
  G3BP1        75.00
  GRWD1       100.00
 HNRNPC       100.00
 HNRNPK        50.00
 HNRNPM       100.00
IGF2BP3         0.00
  KHSRP       100.00
  LARP4        66.67
  NCBP2         0.00
  PCBP2         0.00
   PPIG        50.00
  PRPF4         0.00
  PRPF8        33.33
  PTBP1       100.00
    QKI        25.00
 RBFOX2        40.00
  RBM22        33.33
  SF3A3        33.33
  SF3B4        40.00
  SRSF1        16.67
   TIA1       100.00
  TIAL1       100.00
  U2AF1         0.00
  U2AF2        33.33
  UCHL5       100.00
   XRN2       100.00


In [15]:
# Ignore zeros

def calculate_sign_concordance_no_zeros(cell_line):
    
    # Manipulate dataframes
    print(cell_line)
    t1 = globals()[f"{cell_line}_data"].copy()
    t2 = globals()[f"{cell_line}_glossary"].T.copy()
    t2.columns = t2.columns + 1  # shift 0-5 → 1-6 to match t1
    
    t1.index.name = None
    t1.columns.name = None
    t2.index.name = None
    t2.columns.name = None
    
    print("t1 columns:", t1.columns.tolist())
    print("t2 columns:", t2.columns.tolist())
    
    common_proteins = t1.index.intersection(t2.index)
    t1 = t1.loc[common_proteins]
    t2 = t2.loc[common_proteins]
    
    t1, t2 = t1.align(t2, join="inner", axis=1)
    
    print(f"RBPs in common: {len(common_proteins)}")
    print(f"Positions being compared: {t1.shape[1]}")
    
    results = []
    total_match = 0
    total_valid = 0
    
    for protein in common_proteins:
        row1 = t1.loc[protein]
        row2 = t2.loc[protein]
    
        valid = row1.notna() & row2.notna() & (row1 != 0) & (row2 != 0)  # exclude zeros
        r1 = row1[valid]
        r2 = row2[valid]
    
        if len(r1) == 0:
            results.append({"protein": protein, "concordance": np.nan})
            continue
    
        matches = (np.sign(r1.values) == np.sign(r2.values)).sum()
        n        = len(r1)
        pct      = round(matches / n * 100, 2)
    
        total_match += matches
        total_valid += n
    
        results.append({"protein": protein, "concordance": pct})
    
    concordance_df = pd.DataFrame(results)
    total_concordance = round(concordance_df["concordance"].mean(), 2)
    
    print(f"\nTotal sign concordance across all proteins: {total_concordance}%\n")
    print(concordance_df.to_string(index=False))

In [14]:
K562 = calculate_sign_concordance_no_zeros('K562')

K562
t1 columns: [1, 2, 3, 4, 5, 6]
t2 columns: [1, 2, 3, 4, 5, 6]
RBPs in common: 31
Positions being compared: 6

Total sign concordance across all proteins: 43.01%

protein  concordance
  AGGF1        66.67
 AKAP8L        66.67
    AQR        33.33
  BUD13        66.67
  DDX24         0.00
  DDX3X         0.00
 DROSHA         0.00
 EFTUD2        66.67
   FMR1        33.33
   FXR1        75.00
  GPKOW        66.67
 HNRNPC       100.00
 HNRNPK        50.00
 HNRNPM       100.00
IGF2BP2         0.00
  KHSRP       100.00
  PCBP1         0.00
  PTBP1        50.00
 RBFOX2         0.00
  RBM15        50.00
  RBM22       100.00
  SF3B4        33.33
 SMNDC1         0.00
  SRSF1        33.33
 TARDBP         0.00
   TIA1       100.00
  U2AF1        25.00
  U2AF2        33.33
  UCHL5        83.33
   UPF1         0.00
   YBX3         0.00


In [15]:
HepG2 = calculate_sign_concordance_no_zeros('HepG2')

HepG2
t1 columns: [1, 2, 3, 4, 5, 6]
t2 columns: [1, 2, 3, 4, 5, 6]
RBPs in common: 32
Positions being compared: 6

Total sign concordance across all proteins: 54.14%

protein  concordance
  AKAP1       100.00
    AQR        33.33
 BCLAF1        50.00
  DDX3X        33.33
  DDX59         0.00
 EFTUD2        40.00
  G3BP1        75.00
  GRWD1       100.00
 HNRNPC       100.00
 HNRNPK        75.00
 HNRNPM       100.00
IGF2BP3          NaN
  KHSRP       100.00
  LARP4        66.67
  NCBP2         0.00
  PCBP2         0.00
   PPIG        50.00
  PRPF4         0.00
  PRPF8        33.33
  PTBP1       100.00
    QKI        25.00
 RBFOX2        40.00
  RBM22        33.33
  SF3A3        33.33
  SF3B4        40.00
  SRSF1        16.67
   TIA1       100.00
  TIAL1       100.00
  U2AF1         0.00
  U2AF2        33.33
  UCHL5       100.00
   XRN2       100.00


## Calculate how similar both cell line SHAPs are to each other and how similar the RBP activities are to each other
### To see if the variance in SHAP is similar to the real variance in the data

In [19]:
# Calculates concordance between RBP activity metric across cell lines

In [47]:
def calculate_concordance_activity_metric(df1, df2):
    
    common_proteins = df1.index.intersection(df2.index)
    t1 = df1.loc[common_proteins]
    t2 = df2.loc[common_proteins]
    
    print(f"RBPs in common: {len(common_proteins)}")
    print(f"Positions being compared: {t1.shape[1]}")
    
    results = []
    
    for protein in common_proteins:
        row1 = t1.loc[protein]
        row2 = t2.loc[protein]
    
        valid = row1.notna() & row2.notna() & (row1 != 0) & (row2 != 0)
        r1 = row1[valid]
        r2 = row2[valid]
    
        if len(r1) == 0:
            results.append({"protein": protein, "concordance": np.nan})
            continue
    
        matches = (np.sign(r1.values) == np.sign(r2.values)).sum()
        n       = len(r1)
        pct     = round(matches / n * 100, 2)
    
        results.append({"protein": protein, "concordance": pct})
    
    concordance_df = pd.DataFrame(results)
    total_concordance = round(concordance_df["concordance"].mean(), 2)
    
    print(f"\nTotal sign concordance across all proteins: {total_concordance}%\n")
    print(concordance_df.to_string(index=False))
    
    return concordance_df

In [48]:
activity = calculate_concordance_activity_metric(HepG2_data, K562_data)

RBPs in common: 16
Positions being compared: 6

Total sign concordance across all proteins: 89.06%

protein  concordance
    AQR        100.0
  DDX3X        100.0
 EFTUD2        100.0
 HNRNPC        100.0
 HNRNPK         50.0
 HNRNPM        100.0
  KHSRP         75.0
  PTBP1        100.0
 RBFOX2         50.0
  RBM22         50.0
  SF3B4        100.0
  SRSF1        100.0
   TIA1        100.0
  U2AF1        100.0
  U2AF2        100.0
  UCHL5        100.0


In [49]:
def calculate_concordance_shap_metric(t1, t2):

    t1 = t1.copy().T
    t2 = t2.copy().T

    t1.index.name = None
    t1.columns.name = None
    t2.index.name = None
    t2.columns.name = None

    common_proteins = t1.index.intersection(t2.index)
    t1 = t1.loc[common_proteins]
    t2 = t2.loc[common_proteins]

    t1, t2 = t1.align(t2, join="inner", axis=1)

    print(f"RBPs in common: {len(common_proteins)}")
    print(f"Positions being compared: {t1.shape[1]}")

    results = []

    for protein in common_proteins:
        row1 = t1.loc[protein]
        row2 = t2.loc[protein]

        valid = row1.notna() & row2.notna() & (row1.abs() != 0) & (row2.abs() != 0)
        r1 = row1[valid]
        r2 = row2[valid]

        if len(r1) == 0:
            results.append({"protein": protein, "concordance": np.nan})
            continue

        matches = (np.sign(r1.values) == np.sign(r2.values)).sum()
        n       = len(r1)
        pct     = round(matches / n * 100, 2)

        results.append({"protein": protein, "concordance": pct})

    concordance_df = pd.DataFrame(results)
    total_concordance = round(concordance_df["concordance"].mean(), 2)

    print(f"\nTotal sign concordance across all proteins: {total_concordance}%\n")
    print(concordance_df.to_string(index=False))

    return concordance_df

In [50]:
shap_test = calculate_concordance_shap_metric(HepG2_glossary, K562_glossary)

RBPs in common: 77
Positions being compared: 6

Total sign concordance across all proteins: 66.18%

 protein  concordance
Position       100.00
   AGGF1        50.00
   AKAP1        66.67
     AQR       100.00
   BUD13        83.33
  CSTF2T        50.00
   DDX3X       100.00
   DDX52        50.00
   DDX55        83.33
    DDX6        50.00
   DGCR8        40.00
   DHX30          NaN
  DROSHA        83.33
  EFTUD2        83.33
  EXOSC5       100.00
 FAM120A        40.00
 FASTKD2         0.00
     FTO        50.00
     FUS         0.00
    FXR2        66.67
   GRWD1       100.00
  GTF2F1        50.00
    HLTF         0.00
 HNRNPA1          NaN
  HNRNPC        40.00
  HNRNPK       100.00
  HNRNPL        66.67
  HNRNPM       100.00
  HNRNPU          NaN
HNRNPUL1          NaN
 IGF2BP1        66.67
    ILF3        50.00
   KHSRP        50.00
   LARP4        33.33
   LARP7          NaN
  LIN28B        66.67
   LSM11        50.00
   MATR3       100.00
   NCBP2        75.00
   NOLC1        33.3